# 02b Classical baselines

HAR, log-HAR, GARCH(1,1) and EWMA (λ = 0.94). Every fitted model is estimated on the 2000–2017 training split only, then frozen.
The logic lives in `gru_vol.baselines`; this notebook only runs it and saves the results. The test split is not scored here.

In [ ]:
# Colab setup: clone the private repo, install deps, mount Drive for artifacts.
# Requires a Colab Secret named GITHUB_TOKEN (fine-grained, Contents read-only on this repo).
import base64, os, subprocess, sys
from google.colab import drive, userdata

REPO = "BerkayPolat-lab/gru-currency-volatility-distillation"
REPO_DIR = "/content/gru-currency-volatility-distillation"

def git(*args):
    # Pass the token as an HTTP header through env vars so it never appears in argv,
    # exception messages, or .git/config, and redact it from any error output.
    token = userdata.get("GITHUB_TOKEN").strip()
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"}
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        stderr = result.stderr.replace(token, "***").replace(basic, "***")
        raise RuntimeError(f"git {args[0]} failed:\n{stderr}") from None

if not os.path.exists(REPO_DIR):
    git("clone", "-q", f"https://github.com/{REPO}.git", REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
sys.path.insert(0, f"{REPO_DIR}/model/src")
os.chdir(f"{REPO_DIR}/model")

drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

In [ ]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

In [ ]:
%pip install -q arch

import json
import numpy as np
import pandas as pd

from gru_vol import baselines as B
from gru_vol.data import load_prices, load_examples
from gru_vol.evaluation import mse, high_vol_mse

# --- load: same rows as notebook 02 ---
examples = load_examples()
with open("artifacts/manifests/data_manifest.json") as f:
    MANIFEST = json.load(f)
for name, info in MANIFEST["examples"]["splits"].items():
    assert (examples["split"] == name).sum() == info["examples"], name

prices = load_prices(CFG)
origins = prices.index.get_indexer(examples.index)
assert (origins >= 0).all(), "an example date is missing from the price series"
h = 5

# --- fit on train only, forecast every origin ---
out = B.run_all(prices.to_numpy(), origins, examples["split"].to_numpy(), h=h)
params = out.pop("_params")

r = B.log_returns(prices.to_numpy())
assert np.allclose(B.variance_target(r, origins, h), examples["target"]), "target misaligned"
assert np.allclose(out["HC"], examples["var_60"]), "HC misaligned"

models = ["HC", "EWMA", "HAR", "logHAR", "GARCH"]
for m in models:
    examples[m] = out[m]

g = params["GARCH"]
print(f"GARCH: omega={g['omega']:.3e}  alpha={g['alpha']:.4f}  beta={g['beta']:.4f}  alpha+beta={g['alpha'] + g['beta']:.4f}")
print("HAR coef [const, d, w, m]:", np.round(params["HAR"]["coef"], 4))

# --- score train and validation (test is not scored here) ---
cutoff = MANIFEST["target"]["high_vol_cutoff"]
rows = []
for split in ["train", "validation"]:
    part = examples[examples["split"].eq(split)]
    for m in models:
        hv, n_hv = high_vol_mse(part["target"], part[m], cutoff)
        rows.append({"split": split, "model": m,
                     "mse_e10": mse(part["target"], part[m]) / 1e-10,
                     "high_vol_mse_e10": hv / 1e-10, "high_vol_n": n_hv})
metrics = pd.DataFrame(rows).set_index(["split", "model"])

hc_val = metrics.loc[("validation", "HC"), "mse_e10"]
assert np.isclose(hc_val, 2.571, atol=1e-3), f"HC val MSE {hc_val:.4f}, expected 2.571"
metrics["gain_vs_HC_pct"] = [
    100 * (1 - row.mse_e10 / metrics.loc[(s, "HC"), "mse_e10"]) for (s, _), row in metrics.iterrows()
]

# --- save locally and to Drive ---
def to_jsonable(p):
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in p.items()}

for folder in ["artifacts/results", f"{ARTIFACTS}/results"]:
    os.makedirs(folder, exist_ok=True)
    examples[["split", "target"] + models].to_csv(f"{folder}/baseline_predictions.csv", index_label="date")
    metrics.reset_index().to_csv(f"{folder}/baseline_metrics.csv", index=False)
    with open(f"{folder}/baseline_params.json", "w") as f:
        json.dump({k: to_jsonable(v) for k, v in params.items()}, f, indent=2)

metrics.round(4)